# Fine-tuning do BERTimbau (Kaggle ou Colab)

Este notebook só prepara o ambiente e chama `src/train_bert.py`. Toda a lógica está no repositório, testada.

**Antes de rodar:**
1. Ligue a **GPU** (Kaggle: Settings > Accelerator > GPU T4 x2 ou P100. Colab: Runtime > Change runtime type > T4).
2. Ligue a **internet** (Kaggle: Settings > Internet; exige telefone verificado). O dataset é baixado do GitHub e o modelo do Hugging Face.
3. Crie o segredo **`HF_TOKEN`** com um token de *escrita* do Hugging Face (Kaggle: Add-ons > Secrets. Colab: ícone de chave na barra lateral). Nunca cole o token no notebook.
4. Troque `HUB_REPO` abaixo pelo seu usuário. O repositório do modelo é criado como **privado**.
5. O repositório do **código** no GitHub precisa estar **público** e com o `src/train_bert.py` já enviado (o notebook faz `git clone`). Se for privado, o clone falha.

No Kaggle, use **Save & Run All (Commit)** para rodar em segundo plano sem depender da aba aberta.

In [ ]:
import torch

assert torch.cuda.is_available(), "Sem GPU: ligue o acelerador nas configurações do notebook."
print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
import os
import subprocess

REPO_URL = "https://github.com/arthurvelloso/sentimento-pt.git"
HUB_REPO = "SEU_USUARIO_HF/bertimbau-sentimento-b2w"  # <- troque pelo seu usuário

if not os.path.exists("sentimento-pt"):
    clone = subprocess.run(
        ["git", "clone", REPO_URL],
        env={**os.environ, "GIT_TERMINAL_PROMPT": "0"},  # falha rápido em vez de pedir senha
    )
    if clone.returncode != 0:
        raise RuntimeError(
            "Não foi possível clonar. O repositório é privado? Torne-o público ou use um token."
        )
os.chdir("sentimento-pt")
print(subprocess.run(["git", "log", "--oneline", "-1"], capture_output=True, text=True).stdout)

In [ ]:
# Só o que falta: o torch, o pandas e o scikit-learn já vêm instalados nas duas plataformas.
# Não reinstale o torch (baixaria outra build, sem a CUDA certa).
!pip install -q -r requirements-train.txt

In [ ]:
# Lê o token dos segredos da plataforma, sem imprimir nada.
try:
    from kaggle_secrets import UserSecretsClient

    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
except ImportError:
    from google.colab import userdata

    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")

assert HUB_REPO.split("/")[0] != "SEU_USUARIO_HF", (
    "Troque HUB_REPO pelo seu usuário do Hugging Face."
)
print("token carregado:", bool(os.environ.get("HF_TOKEN")))

## 1. Rodada rápida (uns 2 a 5 minutos)

Treina com poucos exemplos só para conferir token, download dos dados e envio ao Hub **antes** da rodada longa. O resultado dela não vale nada (o modelo mal aprendeu); o repositório privado será sobrescrito na rodada completa.

In [ ]:
!python -m src.train_bert --train-subset 2000 --max-eval 500 --epochs 1 --hub-repo $HUB_REPO --results-path /tmp/smoke.json --output-dir /tmp/smoke_out

## 2. Rodada completa

Treino completo, 3 épocas, melhor checkpoint pela macro-F1 de validação. Se passar de cerca de 1 hora, interrompa e rode com `--train-subset 30000`, documentando no README. Para outras seeds: `--seed 1` (grava em `results/bertimbau_seed1.json`).

In [ ]:
!python -m src.train_bert --hub-repo $HUB_REPO

In [ ]:
import json

result = json.load(open("results/bertimbau.json", encoding="utf-8"))
print("subconjunto?", result["is_subset"], "| GPU:", result["environment"]["gpu"])
print("tempo de treino (s):", result["training"]["train_seconds"])
for h in result["training"]["history"]:
    print(h)
t = result["test"]
print(f"\nTESTE macro-F1={t['macro_f1']:.4f} acurácia={t['accuracy']:.4f}")
print("latência:", t["latency"])

## Depois

- O `results/bertimbau.json` também foi enviado ao repositório do modelo no Hub (pasta `results/`). Baixe de lá, ou do painel *Output* do Kaggle, e copie para o repositório local.
- Atualize a tabela comparativa do README com os números do JSON.
- O modelo foi treinado em dados **CC BY-NC-SA 4.0** (B2W-Reviews01): ao tornar o repositório público, use a licença `cc-by-nc-sa-4.0` e cite a B2W no model card.